In [48]:
# Load Data
import cv2
import numpy as np
import matplotlib.pyplot as plt
from scipy.ndimage import rotate
from ATLAS.Utils.analysisu import *

adata = anndata.read_h5ad('/scratchdata1/MouseBrainAtlases/WTM01/Layer/cell_layer.h5ad')
adata = adata[adata.obs['Slice']=='WTM01_7.8']
adata.obs.iloc[0]


In [5]:
# Load Mask
import torch.nn.functional as F

mask = torch.load('/scratchdata1/Images2024/Zach/MouseBrainAtlas/WTM01_3.2.A_2.2.B_1.2.C_6.2.D_5.2.E_4.2.F_2024Apr08/Processing_2024May28/WellA-Section3/mask/mask.pt')
mask.shape
# Apply max pooling with a kernel size and stride of 2
downsampled_mask = F.max_pool2d(mask.unsqueeze(0).unsqueeze(0), kernel_size=2, stride=2)

# Remove the batch and channel dimensions
downsampled_mask = downsampled_mask.squeeze(0).squeeze(0)
downsampled_mask = downsampled_mask.numpy().astype(int)
mask_x,mask_y = np.where(downsampled_mask>0)
mask_idxes = downsampled_mask[mask_x,mask_y]
np.unique(mask_idxes).shape

downsampled_mask.shape


In [8]:
notebook_path = '/scratchdata1/MouseBrainAtlases/AnalysisNotebooks/'

In [9]:
# Show Signal For each bit for a section
for i in range(adata.X.shape[1]):
    c = adata.layers['normalized'][:,i].copy()
    vmin,vmax = np.percentile(c, [25,99])
    order = np.argsort(c)
    fig, ax = plt.subplots(figsize=[7, 5])
    ax.scatter(adata.obs['ccf_z'][order], adata.obs['ccf_y'][order],s=0.1,marker='x',c=c[order], vmin=vmin, vmax=vmax, cmap='inferno')
    ax.grid(False)
    ax.axis('off')
    ax.axis('equal')
    
    # Set the background to transparent
    fig.patch.set_facecolor('black')
    ax.patch.set_facecolor('black')
    plt.tight_layout()
    plt.savefig(notebook_path+f"Signal_{i}_inferno_black.png", dpi=300, facecolor='black')
    plt.show()

In [39]:
# Gif
import matplotlib.pyplot as plt
import numpy as np
import imageio
import os
from skimage.transform import rescale
from tqdm import trange
import cv2
# Directory containing the images
image_dir = 'images'

# Load and downsample the images
images = []
mask_x = None
mask_y = None
for i in trange(18):
    img = imageio.imread(f"{notebook_path}/Signal_{i}_inferno_black.png")
    img = np.flipud(img)
    if isinstance(mask_x, type(None)):
        mask_x = img[:,:,0:2].max(2).max(0)>0
        mask_y = img[:,:,0:2].max(2).max(1)>0
    img = img[:,mask_x,:]
    img = img[mask_y,:,:]
    img_downsampled = rescale(img, 0.25, anti_aliasing=True, multichannel=True)
    images.append((img_downsampled * 255).astype(np.uint8))

# Function to generate intermediate frames
def generate_intermediate_frames(img1, img2, num_frames=4):
    intermediate_frames = []
    for alpha in np.linspace(0, 1, num_frames):
        blended_img = (1 - alpha) * img1 + alpha * img2
        intermediate_frames.append(blended_img.astype(np.uint8))
    return intermediate_frames

# Create a list to store all frames including intermediate frames
all_frames = []

# Add the first image with persistence
all_frames.extend([images[0]] * 5)

# Generate and add intermediate frames
for i in trange(1, len(images)):
    all_frames.extend(generate_intermediate_frames(images[i-1], images[i]))
    all_frames.extend([images[i]] * 3)

# Save the GIF with fade transitions
imageio.mimsave(f"{notebook_path}/Signal_inferno_fade_hold.gif", all_frames, duration=0.2)

# Save the frames as a movie (MP4)
height, width, layers = all_frames[0].shape
video = cv2.VideoWriter(f"{notebook_path}/Signal_inferno_fade_hold.mp4", cv2.VideoWriter_fourcc(*'mp4v'), 5, (width, height))

for frame in all_frames:
    # Convert RGB to BGR for OpenCV
    frame_bgr = cv2.cvtColor(frame, cv2.COLOR_RGB2BGR)
    video.write(frame_bgr)

video.release()

In [73]:
# Zoomed in view of Signal

from matplotlib.colors import LinearSegmentedColormap
# Create a custom colormap based on 'jet'
jet = plt.cm.get_cmap('inferno', 256)
new_colors = jet(np.linspace(0, 1, 256))
new_colors[0] = np.array([1,1,1, 0])  # Change the first color to black (RGBA)
custom_jet = LinearSegmentedColormap.from_list('inferno', new_colors)

for bit in range(adata.layers['normalized'].shape[1]):

    c = temp_adata.layers['normalized'][:,bit].copy()
    c = np.clip(c, 0, None)
    vmin,vmax = np.percentile(c, (5, 95))

    c = np.clip(c, vmin, vmax)
    converter = dict(zip(np.array(temp_adata.obs['label']).astype(int),c))
    new_values = np.array(pd.Series(mask_idxes).map(converter).values)
    new_values[np.isnan(new_values)] = 0

    img = np.zeros_like(downsampled_mask).astype(float)
    img[mask_x,mask_y] = new_values
    img = img[:,img.max(0)>0]
    img = img[img.max(1)>0,:]
    fig, ax = plt.subplots(figsize=[12, 1])
    ax.imshow(img[150:,70:],vmin=vmin-0.1,vmax=vmax,cmap=custom_jet)
    ax.grid(False)
    ax.axis('off')
    ax.axis('equal')

    # Set the background to transparent
    fig.patch.set_alpha(0.0)
    ax.patch.set_alpha(0.0)
    plt.savefig(notebook_path+f"Signal_{bit}_masked_inferno.png", dpi=300)
    plt.show()